In [1]:
# ---------------------------------------------------------
# Cell 1: Import Libraries
# ---------------------------------------------------------
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import ipywidgets as widgets
from IPython.display import display, clear_output
import urllib.request
import zipfile
import os
import warnings
warnings.filterwarnings('ignore') # Keeps our presentation clean from red warning text

print("Step 1 Complete: All libraries imported successfully!")

Step 1 Complete: All libraries imported successfully!


In [7]:
# ---------------------------------------------------------
# Cell 2: Download the MODERN MovieLens Dataset
# ---------------------------------------------------------
def fetch_modern_data():
    url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
    zip_path = "ml-latest-small.zip"
    
    if not os.path.exists("ml-latest-small"):
        print("Downloading modern MovieLens dataset...")
        urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall()
        print("Download complete.")
    else:
        print("Modern dataset found locally.")

fetch_modern_data()

# The modern dataset is cleaner, but removed user demographics for privacy.
ratings = pd.read_csv('ml-latest-small/ratings.csv')
movies = pd.read_csv('ml-latest-small/movies.csv')

# Merge them together
data = pd.merge(ratings, movies, on='movieId')
print(f"Step 2 Complete: Loaded {len(data)} modern movie ratings!")

Download complete.
Step 2 Complete: Loaded 100836 modern movie ratings!


In [8]:
# ---------------------------------------------------------
# Cell 3: Model Training with Genre Extraction
# ---------------------------------------------------------
print("Extracting genres and training the modern model...")

# 1. Calculate user and movie averages
user_avg = data.groupby('userId')['rating'].mean().reset_index().rename(columns={'rating': 'user_avg_rating'})
movie_avg = data.groupby('movieId')['rating'].mean().reset_index().rename(columns={'rating': 'movie_avg_rating'})
data = pd.merge(data, user_avg, on='userId')
data = pd.merge(data, movie_avg, on='movieId')

# 2. Extract Genres! (Turns "Action|Sci-Fi" into separate columns of 1s and 0s)
genre_dummies = data['genres'].str.get_dummies(sep='|')
data = pd.concat([data, genre_dummies], axis=1)

# Get a list of all unique genres for our UI later
all_genres = list(genre_dummies.columns)
if '(no genres listed)' in all_genres: all_genres.remove('(no genres listed)')

# 3. Define our new modern features
features = ['userId', 'movieId', 'user_avg_rating', 'movie_avg_rating'] + all_genres

X = data[features]
y = data['rating']

# 4. Train the model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.1, max_depth=5)
model.fit(X_train, y_train)

# 5. Prepare the Movie Catalog for the UI
catalog_cols = ['movieId', 'title', 'genres', 'movie_avg_rating'] + all_genres
movie_catalog = data[catalog_cols].drop_duplicates()

rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
print(f"Step 3 Complete! Error margin (RMSE): {rmse:.2f} stars.")

Extracting genres and training the modern model...
Step 3 Complete! Error margin (RMSE): 0.79 stars.


In [11]:
# ---------------------------------------------------------
# Cell 4: Interactive UI with Genre Filtering & Inverted Critic Logic
# ---------------------------------------------------------
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- 1. UI Widgets ---
# Updated the tooltip and default value to match the new logic
critic_slider = widgets.FloatSlider(
    value=3.0, min=1.0, max=5.0, step=0.1, 
    description='Critic Level:', 
    tooltip='1.0 = Loves everything, 5.0 = Hates everything (Very Strict)'
)

# A multi-select box for genres (Hold Cmd/Ctrl to select multiple)
genre_selector = widgets.SelectMultiple(
    options=all_genres,
    value=['Action', 'Sci-Fi'], 
    description='Genres:',
    rows=6
)

predict_btn = widgets.Button(description="Show My Movies!", button_style='success')
output_area = widgets.Output()

# --- 2. Prediction Logic ---
def generate_modern_predictions(b):
    with output_area:
        clear_output()
        selected_genres = list(genre_selector.value)
        
        if not selected_genres:
            print("⚠️ Please select at least one genre!")
            return
            
        print(f"Searching for modern {', '.join(selected_genres)} movies...\n")
        
        # Make a copy of the catalog to predict on
        live_data = movie_catalog.copy()
        live_data['userId'] = 999999
        
        # NEW: Invert the critic slider logic! 
        # High slider value (5.0) becomes a low average rating (1.0)
        actual_avg_rating = 6.0 - critic_slider.value
        live_data['user_avg_rating'] = actual_avg_rating
        
        # Predict ratings for ALL movies
        live_data['predicted_rating'] = model.predict(live_data[features])
        
        # Filter the results to ONLY show movies that match the user's chosen genres
        pattern = '|'.join(selected_genres)
        filtered_movies = live_data[live_data['genres'].str.contains(pattern, case=False, regex=True)]
        
        # Sort and get top 5
        top_5 = filtered_movies.sort_values('predicted_rating', ascending=False).head(5)
        
        print(f"🍿 Your Top 5 Personalized Recommendations (Critic Level: {critic_slider.value}/5.0):")
        for index, row in top_5.iterrows():
            stars = round(row['predicted_rating'], 2)
            print(f"⭐️ {stars} / 5.0 --> {row['title']}")

# --- 3. Display ---
predict_btn.on_click(generate_modern_predictions)

ui_layout = widgets.VBox([
    widgets.HTML("<h2>🍿 Movie Matchmaker</h2>"),
    widgets.HTML("<p>Select your critic level (higher = stricter) and favorite genres.</p>"),
    critic_slider,
    genre_selector,
    widgets.HTML("<br>"),
    predict_btn,
    widgets.HTML("<hr>"),
    output_area
])

display(ui_layout)